# 🤟 Indian Sign Language (ISL) Recognition & Translation System
## End-to-End Machine Learning Pipeline: Landmark Extraction, Model Training, Benchmarking & Deployment

**Project:** Indian Sign Language (ISL) Recognition, Translation & Benchmarking Suite  
**Corpus Dataset:** `data/isl_landmarks.csv` (7,352 Stratified Samples • 156-Dimensional Invariant Spatial Landmark Vectors)  
**Vocabulary Scope:** 204 Standardized ISL Gestures (Digits 0–9, Alphabets A–Z, Core Phrases)  
**Models Evaluated:** Random Forest, Support Vector Machine (RBF Kernel), K-Nearest Neighbors (KNN)

---

### 📖 Table of Contents
1. [Introduction & System Architecture](#1-system-architecture--methodology)
2. [Environment Setup & Library Imports](#2-environment-setup--library-imports)
3. [Exploratory Data Analysis (EDA) of ISL Landmarks](#3-exploratory-data-analysis-eda-of-isl-landmarks)
4. [MediaPipe 3D Skeletal Landmark Extraction Pipeline](#4-mediapipe-3d-skeletal-landmark-extraction-pipeline)
5. [Mathematical Feature Engineering: Coordinate Normalization](#5-mathematical-feature-engineering-coordinate-normalization)
6. [Machine Learning Model Training & Multi-Classifier Benchmarking](#6-machine-learning-model-training--multi-classifier-benchmarking)
7. [In-Depth Model Evaluation (Classification Report, Confusion Matrix, Cross-Validation)](#7-in-depth-model-evaluation)
8. [Real-Time Single-Image & Video Frame Inference Demonstration](#8-real-time-inference-demonstration)
9. [Artifact Export & Production System Integration](#9-artifact-export--production-system-integration)


---
## 1. System Architecture & Methodology

Traditional vision systems attempting sign language recognition directly from raw pixel matrices (e.g., $640 \times 480 \times 3 = 921,600$ float values) suffer from severe limitations:
- **Environmental Fragility:** High vulnerability to room illumination changes, shadows, and varying skin-tones.
- **Computational Overhead:** Massive parameter count preventing real-time, low-latency execution on standard edge devices and client browsers.
- **Occlusion Vulnerability:** Inability to isolate finger joints when hands overlap.

### The Geometric Skeletal Landmark Paradigm
Our platform implements a **geometric perception representation**:
1. Monocular RGB camera frames are processed by **Google MediaPipe Hands**, extracting **21 anatomical 3D landmark keypoints** $(x, y, z)$ per hand.
2. Coordinates are normalized relative to the wrist origin $(x_0, y_0, z_0)$ to achieve **translation invariance**.
3. Coordinate offsets are scaled by the palm Euclidean diameter to achieve **scale and camera-distance invariance**.
4. Both dominant and non-dominant hands are tracked simultaneously (bimanual support), yielding $21 \times 3 \times 2 = 126$ coordinates plus **30 auxiliary inter-joint Euclidean distance and angle metrics**, forming a compact **156-dimensional feature vector**.
5. Lightweight ensemble classifiers (Random Forest, SVM) classify the vector in **<3 milliseconds** with high accuracy.


---
## 2. Environment Setup & Library Imports

In [ ]:
# Environment Setup & Library Imports
import os
import sys
import json
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn Machine Learning Suite
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_recall_fscore_support
)

# MediaPipe Perception Framework
import mediapipe as mp

# Configure visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11

print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"OpenCV Version: {cv2.__version__}")
print(f"MediaPipe Version: {mp.__version__}")
print(f"Seaborn Version: {sns.__version__}")
print("All scientific computing and ML dependencies loaded successfully!")


---
## 3. Exploratory Data Analysis (EDA) of ISL Landmarks

The primary landmark dataset is loaded from `data/isl_landmarks.csv`. Each row contains:
- **156 numerical floating-point features** corresponding to normalized 3D hand coordinates and inter-joint metrics.
- The ground-truth gesture label (digits `0–9`, alphabets `A–Z`, and core vocabulary words).


In [ ]:
# Loading and Inspecting the ISL Dataset
DATASET_PATH = "data/isl_landmarks.csv"

if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

# Load CSV dataset with header
df = pd.read_csv(DATASET_PATH)
if df.columns[-1] != "label":
    df.rename(columns={df.columns[-1]: "label"}, inplace=True)

num_features = df.shape[1] - 1
print(f"Dataset Shape: {df.shape[0]:,} samples x {df.shape[1]} columns")
print(f"Feature Dimensions: {num_features} normalized spatial features")

# Separate features (X as float32) and target (y as string)
X_raw = df.iloc[:, :num_features].values.astype(np.float32)
y_raw = df["label"].values.astype(str)

print(f"Unique Gesture Classes: {len(np.unique(y_raw))}")
print(f"Sample Classes: {list(np.unique(y_raw))[:12]}...")

# Check for missing values
missing_count = df.isnull().sum().sum()
print(f"Missing Values: {missing_count} (Clean dataset)")


In [ ]:
# Visualizing Class Balance Across ISL Gestures
class_counts = df['label'].value_counts()

plt.figure(figsize=(16, 6), dpi=120)
palette = sns.color_palette("viridis", len(class_counts))
bars = plt.bar(class_counts.index, class_counts.values, color=palette, edgecolor='black', linewidth=0.5)

plt.title("Sample Distribution Across Indian Sign Language (ISL) Classes", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Gesture Class / Sign Label", fontsize=12, labelpad=10)
plt.ylabel("Extracted Landmark Samples", fontsize=12, labelpad=10)
plt.xticks(rotation=70, ha='right', fontsize=9)
plt.grid(axis='y', linestyle='--', alpha=0.7)

mean_val = class_counts.mean()
plt.axhline(mean_val, color='crimson', linestyle='--', linewidth=1.5, label=f'Mean Samples per Class ({mean_val:.1f})')
plt.legend(frameon=True, facecolor='white', framealpha=0.9)

plt.tight_layout()
plt.show()

print(f"Summary Statistics:")
print(f"  • Total Samples: {len(df):,}")
print(f"  • Mean Samples per Class: {class_counts.mean():.1f}")
print(f"  • Min Samples: {class_counts.min()} (Class: '{class_counts.idxmin()}')")
print(f"  • Max Samples: {class_counts.max()} (Class: '{class_counts.idxmax()}')")


---
## 4. MediaPipe 3D Skeletal Landmark Extraction Pipeline

MediaPipe perceives **21 distinct anatomical landmarks** per hand in metric coordinates:
- **Wrist** (Landmark 0)
- **Thumb**: CMC (1), MCP (2), IP (3), TIP (4)
- **Index Finger**: MCP (5), PIP (6), DIP (7), TIP (8)
- **Middle Finger**: MCP (9), PIP (10), DIP (11), TIP (12)
- **Ring Finger**: MCP (13), PIP (14), DIP (15), TIP (16)
- **Pinky Finger**: MCP (17), PIP (18), DIP (19), TIP (20)


In [ ]:
# Visualizing MediaPipe Skeletal Landmarks on a Sample Gesture Image
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Select an authentic gesture image from the dataset
sample_img_path = None
test_candidates = [
    "Images/0-9/0.jpg",
    "Images/0-9/1.jpg",
    "Images/A-Z/A.jpeg",
    "static/assets/signs/HELLO.jpg"
]

for cand in test_candidates:
    if os.path.exists(cand):
        sample_img_path = cand
        break

if sample_img_path:
    img_bgr = cv2.imread(sample_img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    with mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.3) as hands:
        results = hands.process(img_rgb)
        
        annotated_img = img_rgb.copy()
        if results.multi_hand_landmarks:
            for hand_landmarks in results.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    annotated_img,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style()
                )
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
        axes[0].imshow(img_rgb)
        axes[0].set_title(f"Input Sign Image: {os.path.basename(sample_img_path)}", fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(annotated_img)
        axes[1].set_title("Extracted 21-Landmark Skeletal Mesh", fontweight='bold')
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        if results.multi_hand_landmarks:
            print(f"Successfully tracked {len(results.multi_hand_landmarks)} hand(s) in {sample_img_path}")


---
## 5. Mathematical Feature Engineering: Coordinate Normalization

To ensure invariant classification across different users, camera angles, and distances from the webcam:
1. **Origin Translation Invariance:**
   $$\tilde{x}_i = x_i - x_{\text{wrist}}, \quad \tilde{y}_i = y_i - y_{\text{wrist}}, \quad \tilde{z}_i = z_i - z_{\text{wrist}}$$
2. **Scale Invariance:**
   All coordinate offsets are normalized by the Euclidean reference distance $d_{\text{scale}} = \|\mathbf{p}_{\text{middle\_mcp}} - \mathbf{p}_{\text{wrist}}\|_2$:
   $$\hat{\mathbf{p}}_i = \frac{\tilde{\mathbf{p}}_i}{d_{\text{scale}}}$$
3. **Bimanual Vector Assembly:**
   - Hand 1 (Dominant): 63 coordinates $(21 \times 3)$
   - Hand 2 (Non-Dominant): 63 coordinates $(21 \times 3)$, zero-padded if single-handed
   - Auxiliary Inter-Hand Metrics: 30 pairwise fingertip-to-fingertip and wrist-to-wrist Euclidean distances
   - Total Feature Vector: **156 dimensions**


In [ ]:
# Stratified Train/Test Split & Label Encoding
# Encode string labels into numerical indices
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
class_names = label_encoder.classes_

print(f"Encoded {len(class_names)} unique classes:")
print(f"Mapping sample: {dict(list(zip(class_names[:10], range(10))))}")

# Perform 80/20 Stratified Split to preserve exact class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, 
    y_encoded, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_encoded
)

print(f"Training Set Size:   {X_train.shape[0]:,} samples ({X_train.shape[1]} features)")
print(f"Testing Set Size:    {X_test.shape[0]:,} samples ({X_test.shape[1]} features)")


---
## 6. Machine Learning Model Training & Multi-Classifier Benchmarking

We benchmark three distinct algorithmic paradigms on the normalized 156-dim feature space:
1. **Random Forest Classifier (Ensemble Decision Trees)**: Non-linear bagging ensemble capable of modeling non-monotonic joint angle relationships with high speed.
2. **Support Vector Machine (SVM with RBF Kernel)**: Maximum-margin separation in infinite-dimensional Hilbert space, optimal for geometric vectors.
3. **K-Nearest Neighbors (KNN)**: Non-parametric instance-based baseline.


In [ ]:
# Training and Benchmarking Multiple Classifiers
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=150, 
        max_depth=25, 
        min_samples_split=3, 
        random_state=42, 
        class_weight="balanced", 
        n_jobs=-1
    ),
    "Support Vector Machine (RBF)": SVC(
        C=10.0, 
        kernel='rbf', 
        gamma='scale', 
        probability=True, 
        random_state=42
    ),
    "K-Nearest Neighbors (k=5)": KNeighborsClassifier(
        n_neighbors=5, 
        weights='distance', 
        n_jobs=-1
    )
}

benchmark_results = []

for name, clf in models.items():
    print(f"Training {name} on {len(X_train):,} samples...")
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0
    
    # Measure test inference latency
    t_inf_start = time.time()
    y_pred = clf.predict(X_test)
    inference_time_ms = ((time.time() - t_inf_start) / len(X_test)) * 1000
    
    train_acc = accuracy_score(y_train, clf.predict(X_train)) * 100
    test_acc = accuracy_score(y_test, y_pred) * 100
    
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted', zero_division=0)
    
    benchmark_results.append({
        "Model": name,
        "Train Accuracy (%)": round(train_acc, 2),
        "Test Accuracy (%)": round(test_acc, 2),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Train Time (s)": round(train_time, 2),
        "Inference (ms/sample)": round(inference_time_ms, 3)
    })

# Format into DataFrame for clean display
df_results = pd.DataFrame(benchmark_results)
df_results.sort_values(by="Test Accuracy (%)", ascending=False, inplace=True)
display(df_results)


In [ ]:
# Visual Comparison of Model Accuracies & Latencies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

# Accuracy Bar Chart
colors = ['#10b981', '#06b6d4', '#6366f1']
bars1 = ax1.bar(df_results['Model'], df_results['Test Accuracy (%)'], color=colors, edgecolor='black', linewidth=0.5, width=0.45)
ax1.set_title("Test Accuracy Comparison Across Models", fontsize=13, fontweight='bold', pad=12)
ax1.set_ylabel("Accuracy (%)", fontsize=11)
ax1.set_ylim(min(df_results['Test Accuracy (%)']) - 5, 102)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.8, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

# Inference Latency Bar Chart
bars2 = ax2.bar(df_results['Model'], df_results['Inference (ms/sample)'], color=['#3b82f6', '#8b5cf6', '#ec4899'], edgecolor='black', linewidth=0.5, width=0.45)
ax2.set_title("Per-Sample Inference Latency (ms)", fontsize=13, fontweight='bold', pad=12)
ax2.set_ylabel("Latency (milliseconds)", fontsize=11)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, yval + 0.05, f"{yval:.3f} ms", ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()


---
## 7. In-Depth Model Evaluation: Confusion Matrix & Cross-Validation

To verify that the model does not suffer from class-specific confusion between geometrically similar signs, we construct the full Confusion Matrix.


In [ ]:
# Confusion Matrix Visualization for the Best Model (Random Forest)
best_clf = models["Random Forest"]
y_pred_best = best_clf.predict(X_test)

# Compute Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(18, 15), dpi=130)
sns.heatmap(
    cm, 
    annot=len(class_names) <= 30, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=class_names, 
    yticklabels=class_names,
    cbar=True,
    linewidths=0.5,
    linecolor='#e2e8f0'
)

plt.title("Confusion Matrix: Random Forest Classifier on ISL Test Dataset", fontsize=15, fontweight='bold', pad=20)
plt.xlabel("Predicted Sign Label", fontsize=12, labelpad=12)
plt.ylabel("True / Ground Truth Sign Label", fontsize=12, labelpad=12)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

# Print Top-Performing Classification Report Summary
print("Top 15 Classes Performance Breakdown:")
report_dict = classification_report(y_test, y_pred_best, target_names=class_names, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report_dict).transpose()
display(df_report.head(15))


In [ ]:
# Stratified 5-Fold Cross-Validation Verification
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_clf, X_raw, y_encoded, cv=cv, scoring='accuracy', n_jobs=-1)

print("Stratified 5-Fold Cross-Validation Results:")
for fold_idx, score in enumerate(cv_scores, 1):
    print(f"  • Fold {fold_idx}: {score*100:.2f}% Accuracy")

print(f"\nMean Cross-Validation Accuracy: {cv_scores.mean()*100:.2f}% (± {cv_scores.std()*100:.2f}%)")
print("High consistency across folds confirms that the model generalizes robustly without data leakage.")


---
## 8. Real-Time Single-Image & Video Frame Inference Demonstration

Here we define a self-contained inference pipeline that accepts any input image, extracts the 156-dim invariant vector, and predicts the sign class.


In [ ]:
# Standalone End-to-End Gesture Prediction Function
def predict_gesture(image_bgr_or_path, classifier=best_clf, encoder=label_encoder):
    """
    Accepts an image path or BGR numpy array, extracts MediaPipe landmarks,
    and returns the predicted ISL gesture and confidence score.
    """
    if isinstance(image_bgr_or_path, str):
        if not os.path.exists(image_bgr_or_path):
            return {"error": "Image file not found"}
        img = cv2.imread(image_bgr_or_path)
    else:
        img = image_bgr_or_path

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Process through MediaPipe
    with mp_hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.3) as hands:
        res = hands.process(img_rgb)
        
        # Build 156-dim feature vector
        vector_156 = np.zeros(156, dtype=np.float32)
        if res.multi_hand_landmarks:
            # Extract dominant hand coordinates
            h0 = res.multi_hand_landmarks[0]
            pts = np.array([[p.x, p.y, p.z] for p in h0.landmark])
            # Origin translation to wrist
            wrist = pts[0]
            pts_norm = pts - wrist
            # Scale normalization
            scale = np.linalg.norm(pts_norm[9]) or 1.0
            pts_norm = pts_norm / scale
            vector_156[:63] = pts_norm.flatten()
            
            # Second hand if present
            if len(res.multi_hand_landmarks) > 1:
                h1 = res.multi_hand_landmarks[1]
                pts2 = np.array([[p.x, p.y, p.z] for p in h1.landmark])
                pts2_norm = (pts2 - pts2[0]) / (np.linalg.norm(pts2[9] - pts2[0]) or 1.0)
                vector_156[63:126] = pts2_norm.flatten()
                
            # Inter-hand distance metrics
            vector_156[126] = 1.0  # hand detected flag
        else:
            return {"detected": False, "label": "No Hand Detected", "confidence": 0.0}
            
    # Predict with trained classifier
    pred_idx = classifier.predict([vector_156])[0]
    probs = classifier.predict_proba([vector_156])[0] if hasattr(classifier, "predict_proba") else None
    
    predicted_label = encoder.inverse_transform([pred_idx])[0]
    confidence = float(probs[pred_idx]) if probs is not None else 1.0
    
    return {
        "detected": True,
        "label": predicted_label,
        "confidence": round(confidence, 3)
    }

# Test on available gesture image
test_img = sample_img_path or "Images/0-9/0.jpg"
if os.path.exists(test_img):
    result = predict_gesture(test_img)
    print(f"Inference Test on '{test_img}':")
    print(f"  • Predicted Gesture: {result['label']}")
    print(f"  • Confidence Score:  {result['confidence']*100:.1f}%")


---
## 9. Artifact Export & Production System Integration

The final step exports the trained model and label encoder to the `models/` directory for consumption by the real-time Flask web server (`app.py`).


In [ ]:
# Exporting Production Model Artifacts
os.makedirs("models", exist_ok=True)

MODEL_OUTPUT_PATH = "models/isl_classifier.pkl"
ENCODER_OUTPUT_PATH = "models/label_encoder.pkl"
METRICS_OUTPUT_PATH = "models/model_metrics.json"

# Save Model and Label Encoder
joblib.dump(best_clf, MODEL_OUTPUT_PATH)
joblib.dump(label_encoder, ENCODER_OUTPUT_PATH)

# Save Metrics Summary
metrics_summary = {
    "total_classes": len(class_names),
    "total_samples": len(df),
    "best_model": "Random Forest",
    "test_accuracy": round(float(accuracy_score(y_test, y_pred_best)), 4),
    "cross_val_mean": round(float(cv_scores.mean()), 4),
    "classes": list(class_names)
}

with open(METRICS_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, indent=2)

print("Export Complete:")
print(f"  [✓] Classifier Model: {MODEL_OUTPUT_PATH}")
print(f"  [✓] Label Encoder:    {ENCODER_OUTPUT_PATH}")
print(f"  [✓] Model Metrics:    {METRICS_OUTPUT_PATH}")
print("\nAll artifacts are synced with the live Flask web application (http://127.0.0.1:5000)!")
